# 16 — TF-IDF Deep Dive

**Goal:** Understand TF-IDF weighting — the most popular text representation in classical ML.

## 1. TF-IDF Manually

In [ ]:
import numpy as np
from collections import Counter
import math
docs = [
    "Python is a programming language for data science",
    "Python is used in machine learning and data science",
    "Java is another programming language",
]
# Manual TF-IDF
def tf(word, doc):
    words = doc.lower().split()
    count = words.count(word.lower())
    return count / len(words) if len(words) > 0 else 0

def idf(word, docs):
    n_containing = sum(1 for d in docs if word.lower() in d.lower().split())
    return math.log(len(docs) / (1 + n_containing)) + 1

word = "python"
print(f"TF-IDF for '{word}':")
for i, d in enumerate(docs):
    tfidf = tf(word, d) * idf(word, docs)
    print(f"  Doc {i+1}: TF={tf(word,d):.4f} x IDF={idf(word,docs):.4f} = {tfidf:.4f}")

## 2. TF-IDF with scikit-learn

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
vec = TfidfVectorizer(stop_words="english", max_features=10)
X = vec.fit_transform(docs)
print(f"Vocabulary: {vec.get_feature_names_out()}")
print("\nTF-IDF Matrix:")
for i in range(len(docs)):
    scores = [(w, X[i, j]) for w, j in zip(vec.get_feature_names_out(), range(X.shape[1])) if X[i, j] > 0]
    scores.sort(key=lambda x: -x[1])
    print(f"  Doc {i+1}: {[(w, round(s,3)) for w, s in scores[:5]]}")

## 3. TF-IDF for Resume-JD Matching

In [ ]:
resume = "python tensorflow machine learning nlp data science experience"
jd = "python machine learning tensorflow deep learning data science required"
vec = TfidfVectorizer(stop_words="english")
X = vec.fit_transform([resume, jd])
features = vec.get_feature_names_out()
print(f"Shared keywords: {set(features)}")
from sklearn.metrics.pairwise import cosine_similarity
sim = cosine_similarity(X[0:1], X[1:2])[0][0]
print(f"Resume-JD TF-IDF similarity: {sim:.3f}")

## Key Insight: TF-IDF downweights common words ('experience', 'data') and highlights distinctive ones (skills, tech).